# 04 - Poison-fraction sweep (pre-registered) + optional all-labels runs
Pre-registration: `experiments/prereg_sweep.md` (committed before any run). This notebook:
1. copies the prepared Stage 1/1b files into `/kaggle/working/nids_out` and checks that their sha256 equal the ones `main` used;
2. runs `repro_s42` (the `main` settings again) and compares its **validation decisions** with `main`;
3. runs the 12 sweep runs `sweep_f{0.75,0.9,0.95,1.0}_s{42,43,44}`;
4. makes the **validation-only** decision table (`s6_sweep.py --decisions`) → **STOP**: send `decisions.csv`, it is committed to git;
5. only then the test table (`s6_sweep.py --test`);
6. (optional, off) part B: the all-labels schedule, CII then CI, seed 42;
7. zips only the new outputs.

`main` is never touched: it is read from the read-only results dataset, and `s3_continual.py` refuses to overwrite any completed run.

**Inputs (Add Input):**
| input | what it provides |
|---|---|
| code dataset `dvdhere/nids-code` | the cleaned parquet splits (`results_01`, found by searching for `clean_train.parquet`) and, after a new dataset version, the code |
| results dataset `dvdhere/nids-results` | the completed `main` run (`runs/cii/main`), for the reproduction check |
| (or) Internet ON + `git clone` | the newest code from GitHub (cell 1 prefers it) |

Accelerator: **GPU T4**.

In [ ]:
# 1. Locate code, prepared files and main; copy prepared files to a writable folder
import glob, hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

CLONE = Path("/kaggle/working/code")
if not CLONE.exists() and os.environ.get("NIDS_CLONE", "1") == "1":
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DevChi1l/cicddos2019-continual-nids",
                    str(CLONE)], check=False)
# the marker s6_sweep.py exists only in the new code, so an old code dataset is never picked by mistake
CODE_DIR = os.environ.get("NIDS_CODE") or next(
    (str(Path(p).parent) for p in [str(CLONE / "s6_sweep.py")] +
     sorted(glob.glob("/kaggle/input/**/s6_sweep.py", recursive=True)) if Path(p).exists()), None)
assert CODE_DIR, "new code not found: turn Internet ON (git clone) or attach a code dataset version with s6_sweep.py"
sys.path.insert(0, CODE_DIR)
import s5_explain as X
from common import load_json

git = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code:", CODE_DIR, "| git commit:", git or "(not a git checkout) s3 sha256 " +
      hashlib.sha256(Path(CODE_DIR, "s3_continual.py").read_bytes()).hexdigest()[:16])

WORK = Path(os.environ.get("NIDS_WORK", "/kaggle/working/nids_out"))
PREP = X.find_dir("clean_train.parquet", "NIDS_PREPARED")   # results_01 inside the code dataset
assert PREP, "clean_train.parquet not found under /kaggle/input: attach the code dataset that contains results_01"
if PREP.resolve() != WORK.resolve():
    shutil.copytree(PREP, WORK, dirs_exist_ok=True, ignore=shutil.ignore_patterns("runs"))
MAIN_OUT = X.find_dir("runs/cii/main/run_manifest.json", "NIDS_RESULTS")
MAIN = MAIN_OUT / "runs" / "cii" / "main" if MAIN_OUT else None
print("prepared:", PREP, "->", WORK, "\nmain run:", MAIN)

# the copied files must be exactly the files main was built from
if MAIN:
    want = load_json(MAIN / "config.json")["inputs"]
    def sha(p):
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
        return h.hexdigest()
    bad = {n: "missing" if not (WORK / n).exists() else "different" for n in want
           if not (WORK / n).exists() or sha(WORK / n) != want[n]}
    assert not bad, f"prepared files differ from main's inputs: {bad}"
    print(f"all {len(want)} Stage 1/1b inputs match main's sha256")
else:
    print("WARNING: main not found - the reproduction check (H0) will be skipped")

def run(script, *args):
    cmd = [sys.executable, str(Path(CODE_DIR) / script), *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    subprocess.run(cmd, check=True, cwd=CODE_DIR)

## 2. GPU and dry run (loads everything, checks it against Stage 1b, trains nothing)

In [ ]:
import torch
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4"
print(torch.cuda.get_device_name(0), torch.__version__)
SCENARIO = "cii"
run("s3_continual.py", "--scenario", SCENARIO, "--dry-run")

## 3. Reproduction: `repro_s42` = the `main` settings (defaults, seed 42, f = 0.5)
H0: same gate decisions as `main`. On GPU exact equality of every number is not guaranteed; the max difference is printed, not hidden.

In [ ]:
RUNS = WORK / "runs" / SCENARIO
def done(name): return (RUNS / name / "run_manifest.json").exists()
t0 = time.time()
if not done("repro_s42"):
    run("s3_continual.py", "--scenario", SCENARIO, "--run", "repro_s42", "--resume")
run("s4_report.py", "--scenario", SCENARIO, "--run", "repro_s42")
print(f"repro: {time.time() - t0:.0f} s")
if MAIN:
    rows = []
    for arm in ("clean", "poisoned", "poisoned_repair"):
        for m, r in zip(load_json(MAIN / arm / "rounds.json"), load_json(RUNS / "repro_s42" / arm / "rounds.json")):
            rows.append({"arm": arm, "E": m["experience"], "attempt": (m["attempt"], r["attempt"]),
                         "accepted main/repro": (m["accepted"], r["accepted"]),
                         "max |diff| val recall": max(abs(m["val"]["recall"][k] - r["val"]["recall"][k]) for k in m["val"]["recall"])})
    print(pd.DataFrame(rows).to_string(index=False))

## 4. Sweep: 4 poison fractions x 3 seeds (settings fixed by the pre-registration)
Each run trains T/A0 for its seed, then the 3 arms; `--forced-r1` adds **diagnostic** R1s on the clean and poisoned arms (logged in `r1_diag.json`, never used for a decision). If the session dies, rerun this cell: finished runs are skipped and unfinished ones resume.

In [ ]:
FRACS = [0.75, 0.9, 0.95, 1.0]
SEEDS = [42, 43, 44]
FORCED = "soft,threeway,threeway+relabel"
for f in FRACS:
    for s in SEEDS:
        name = f"sweep_f{f}_s{s}"
        if done(name):
            print("done:", name); continue
        t0 = time.time()
        run("s3_continual.py", "--scenario", SCENARIO, "--run", name, "--seed", s, "--poison-frac", f,
            "--forced-r1", FORCED, "--resume")
        run("s4_report.py", "--scenario", SCENARIO, "--run", name)
        print(f"{name}: {time.time() - t0:.0f} s", flush=True)

## 5. Validation-only decision table (no test numbers are read here)

In [ ]:
run("s6_sweep.py", "--scenario", SCENARIO, *(["--main-dir", MAIN] if MAIN else []), "--decisions")
display(pd.read_csv(WORK / "sweep" / "hypotheses.csv"))

## STOP - send `nids_out/sweep/decisions.csv`, `diagnostics.csv` and `hypotheses.csv`
They are committed to git **before** any test number is produced (pre-registration rule 2). After the commit, set `DECISIONS_COMMITTED = True` below and run the next cell.

In [ ]:
DECISIONS_COMMITTED = False
if DECISIONS_COMMITTED:
    run("s6_sweep.py", "--scenario", SCENARIO, "--test")
else:
    print("skipped: commit decisions.csv first, then set DECISIONS_COMMITTED = True")

## Part B (optional, OFF) - all-labels schedule, CII then CI, seed 42
Prepared, **not run** yet: nothing here is a result until this part has run.

Rule (`make_all_labels_schedule.py`): an attack class is included if it has >= 1,000 train and >= 200 val rows. That gives **12 of 14 labels**: Portmap (363 train / 60 val) and WebDDoS (132 / 22) are excluded. E0 = BENIGN + Syn (as in `main`); the other 10 classes, sorted by name, arrive two per experience, so there are 6 experiences. Attack rows per class = the smallest included train count (NetBIOS, 9,312). Benign: CII gets 6 disjoint portions (9,528 + 5 x 9,527); CI puts all 57,163 train benign rows in E0.

New Stage 1b files go to `nids_out/all12/` (`--tag all12`); the files `main` and the sweep used are never touched.

**Protocol deviations to report:** the scaler is refit on the new E0 (a different benign portion), so its protected exclusions and the scored rows differ from `main` (no row-for-row comparison); benign is about 34% of each CII experience instead of 58%; one seed only. A refusal by s1b (more than 5% of a class excluded, or fewer than 5,000 benign val/test rows) stops this part; that is a finding to report, not a reason to loosen the check.

In [ ]:
RUN_ALL_LABELS = False
TAG = "all12"
if RUN_ALL_LABELS:
    sched = WORK / f"{TAG}_schedule.json"
    run("make_all_labels_schedule.py", "--split-manifest", WORK / "split_manifest.json", "--write", sched)
    cap = load_json(sched)["attack_cap"]
    for sc in ("cii", "ci"):
        run("s1b_experiences.py", "--schedule", sched, "--scenario", sc, "--tag", TAG, "--attack-cap", cap)
    import numpy as np
    tr = pd.read_parquet(WORK / "clean_train.parquet", columns=["_rid", "label"]).set_index("_rid")["label"]
    z = {sc: np.load(WORK / TAG / sc / "experience_rids.npz") for sc in ("cii", "ci")}
    for t in range(len(load_json(sched)["experiences"])):
        a = {sc: set(v for v in z[sc][f"e{t}"] if tr[v] != "BENIGN") for sc in z}
        assert a["cii"] == a["ci"], f"E{t}: CI and CII attack rows differ (protocol deviation)"
    b = [set(v for v in z["cii"][f"e{t}"] if tr[v] == "BENIGN") for t in range(len(z["cii"].files))]
    assert all(not (b[i] & b[j]) for i in range(len(b)) for j in range(i + 1, len(b))), "CII benign overlap"
    print("CI and CII share every attack row; CII benign portions are disjoint")
    for sc in ("cii", "ci"):
        name = f"{TAG}_{sc}_s42"
        if not (WORK / "runs" / sc / name / "run_manifest.json").exists():
            run("s3_continual.py", "--scenario", sc, "--tag", TAG, "--run", name, "--forced-r1", FORCED, "--resume")
        run("s4_report.py", "--scenario", sc, "--run", name)
else:
    print("part B skipped (RUN_ALL_LABELS = False)")

## 6. Zip only the new outputs (no parquet, nothing from `main`)

In [ ]:
import zipfile
NAME = "nids_sweep"
dest = WORK.parent / f"{NAME}.zip"
pick = [p for p in WORK.rglob("*") if p.is_file() and p.suffix != ".parquet" and (
    any(part.startswith(("sweep_", "repro_s42", "all12_")) for part in p.relative_to(WORK).parts)
    or p.relative_to(WORK).parts[0] in ("sweep", "all12") or p.name == "all12_schedule.json")]
with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in pick:
        zf.write(p, Path("nids_out") / p.relative_to(WORK))
print(f"{len(pick)} files -> {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
print("Save Version, then download", dest.name, "from the Output tab.")